# Churn Analysis Walkthrough

EDA, feature importance, local SHAP explanations, cumulative gains, and PSI drift simulation for the Telco churn model.

## 1. Dataset profile

Load the Telco schema and inspect churn prevalence and tenure distribution.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
import pandas as pd
raw = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
raw['Churn'].value_counts(normalize=True)

## 2. Feature engineering and model loading

In [ ]:
from src.features import build_features
from src.train import load_model_with_metadata
model, feature_names, onehot = load_model_with_metadata(pathlib.Path('../models/churn_model.joblib'))
df = build_features(raw, fit=False)
X = df.drop(columns=['Churn'])
for col in feature_names:
    if col not in X.columns: X[col] = 0
X = X[feature_names]
y = df['Churn']
X.shape

## 3. SHAP explanations

Use the `/explain` endpoint for customer-level drivers; the saved figure shows global importance.

In [ ]:
from IPython.display import Image, display
display(Image('../reports/shap_summary.png'))

## 4. Cumulative gains

Ranking by churn score shows how many churners are captured as outreach capacity expands.

In [ ]:
from IPython.display import Image, display
display(Image('../reports/cumulative_gains.png'))

## 5. PSI drift simulation

Shift tenure downward to model a newer customer mix and monitor PSI.

In [ ]:
import numpy as np

def compute_psi(expected, actual, bins=10):
    qs = np.quantile(expected, np.linspace(0, 1, bins + 1))
    qs[0], qs[-1] = -np.inf, np.inf
    e = np.histogram(expected, bins=qs)[0] / len(expected)
    a = np.histogram(actual, bins=qs)[0] / len(actual)
    e = np.where(e == 0, 0.0001, e)
    a = np.where(a == 0, 0.0001, a)
    return float(((a - e) * np.log(a / e)).sum())
shifted_tenure = np.maximum(0, raw['tenure'].to_numpy() - 18)
compute_psi(raw['tenure'].to_numpy(), shifted_tenure)